# Neisseria gonorrhoeae – BioAssay Comparison

Objective:
Compare BioAssays obtained from:
1. **PubChem Web UI export**
2. **Aid2Taxid.tsv explicit taxonomy mapping**

We identify assays that appear in the UI but *not* in Aid2Taxid,
and inspect their metadata to understand why.

## 0. Setup

In [6]:
import pandas as pd
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
DATA_RAW = NOTEBOOK_DIR.parent / "data" / "raw"
DATA_PROCESSED = NOTEBOOK_DIR.parent / "data" / "processed"

DATA_RAW

PosixPath('/Users/maria/Documents/Ersilia/PubChem/pubchem-antimicrobial-tasks/data/raw')

## 1. Load UI-exported BioAssays

This file was manually downloaded from:
PubChem → Search “Neisseria gonorrhoeae” → Taxonomy → Actions → BioAssays → Summary (Search Results)

In [2]:
ui_file = DATA_RAW / "PubChem_bioassay_text_Neisseria gonorrhoeae.csv"
df_ui = pd.read_csv(ui_file)

df_ui.head()

,aid,aidtype,aidname,aiddesc,aidcomment,aidsrcid,aidsrcname,aidextid,depdate,aidmdate,...,cellids,targettaxid,cnt,activecnt,anatomyid,anatomy,dois,pmcids,pclids,citations
0,524653,Literature-derived,Antimicrobial activity against Neisseria gonor...,Title: Relation between genetic markers of dru...,Journal: Antimicrob. Agents Chemother._||_Year...,43,ChEMBL,CHEMBL1268727,20110918,20180915,...,NaN,485.0,1,NaN,NaN,NaN,10.1128/aac.01420-07,PMC2415756,9281464.0,"Ilina EN, Vereshchagin VA, Borovskaya AD, Mala..."
1,524669,Literature-derived,Antimicrobial activity against Neisseria gonor...,Title: Relation between genetic markers of dru...,Journal: Antimicrob. Agents Chemother._||_Year...,43,ChEMBL,CHEMBL1268743,20110918,20180915,...,NaN,485.0,1,NaN,NaN,NaN,10.1128/aac.01420-07,PMC2415756,9281464.0,"Ilina EN, Vereshchagin VA, Borovskaya AD, Mala..."
2,524877,Literature-derived,Antimicrobial activity against Neisseria gonor...,Title: Relation between genetic markers of dru...,Journal: Antimicrob. Agents Chemother._||_Year...,43,ChEMBL,CHEMBL1267506,20110918,20180915,...,NaN,485.0,1,NaN,NaN,NaN,10.1128/aac.01420-07,PMC2415756,9281464.0,"Ilina EN, Vereshchagin VA, Borovskaya AD, Mala..."
3,524893,Literature-derived,Antimicrobial activity against Neisseria gonor...,Title: Relation between genetic markers of dru...,Journal: Antimicrob. Agents Chemother._||_Year...,43,ChEMBL,CHEMBL1267605,20110918,20180915,...,NaN,485.0,1,NaN,NaN,NaN,10.1128/aac.01420-07,PMC2415756,9281464.0,"Ilina EN, Vereshchagin VA, Borovskaya AD, Mala..."
4,524638,Literature-derived,Antimicrobial activity against Neisseria gonor...,Title: Relation between genetic markers of dru...,Journal: Antimicrob. Agents Chemother._||_Year...,43,ChEMBL,CHEMBL1268619,20110918,20180915,...,NaN,485.0,3,NaN,NaN,NaN,10.1128/aac.01420-07,PMC2415756,9281464.0,"Ilina EN, Vereshchagin VA, Borovskaya AD, Mala..."


## 2. Load Aid2Taxid.tsv mapping file

In [3]:
aid2taxid_file = DATA_RAW / "Aid2Taxid.tsv"
df_map = pd.read_csv(aid2taxid_file, sep="\t")

df_map.head()

,AID,TaxID
0,1398771,10090
1,1398772,10090
2,1398788,9606
3,1398789,9606
4,1398790,9606


## 3. Identify all Neisseria TaxIDs

Using the taxonomy table created earlier (taxonomy_table.csv)

In [7]:
tax_table = pd.read_csv(DATA_PROCESSED / "taxonomy_table.csv")

neisseria_taxids = tax_table.loc[
    tax_table["Pathogen"] == "Neisseria gonorrhoeae",
    "Taxonomy_ID"
].tolist()

neisseria_taxids

[485]

## 4. Extract AIDs explicitly linked to Neisseria in Aid2Taxid

In [8]:
df_map_neis = df_map[df_map["TaxID"].isin(neisseria_taxids)]

aid2tax_aids = set(df_map_neis["AID"].astype(int))
len(aid2tax_aids)

446

## 5. Extract UI-linked AIDs

In [9]:
ui_aids = set(df_ui["aid"].astype(int))
len(ui_aids)

1019

## 6. Compare the two sets

In [10]:
missing_from_aid2tax = ui_aids - aid2tax_aids
extra_in_aid2tax = aid2tax_aids - ui_aids

summary = pd.DataFrame({
    "Metric": ["UI AIDs", "Aid2Taxid AIDs", "Missing (UI→Aid2Taxid)", "Unexpected (Aid2Taxid→UI)"],
    "Count": [len(ui_aids), len(aid2tax_aids), len(missing_from_aid2tax), len(extra_in_aid2tax)]
})

summary

,Metric,Count
0,UI AIDs,1019
1,Aid2Taxid AIDs,446
2,Missing (UI→Aid2Taxid),574
3,Unexpected (Aid2Taxid→UI),1


## 7. Inspect Missing AIDs (in UI but *not* in Aid2Taxid)

In [20]:
df_missing = df_ui[df_ui["aid"].isin(missing_from_aid2tax)]

df_missing.head(5)

,aid,aidtype,aidname,aiddesc,aidcomment,aidsrcid,aidsrcname,aidextid,depdate,aidmdate,...,cellids,targettaxid,cnt,activecnt,anatomyid,anatomy,dois,pmcids,pclids,citations
214,1982347,Literature-derived,Antimicrobial activity against Neisseria gonor...,Title: Analysis of Structure-Activity Relation...,Journal: J Med Chem_||_Year: 2023_||_Volume: 6...,43,ChEMBL,CHEMBL5335036,20250102,20250102,...,NaN,485.0,28,NaN,NaN,NaN,10.1021/acs.jmedchem.3c00458,NaN,35941841.0,"Scheuplein NJ, Bzdyl NM, Lohr T, Kibble EA, Ha..."
218,1975741,Literature-derived,Antibacterial activity against resistant Neiss...,Title: Synthesis of novobiocin derivatives and...,Journal: Bioorg Med Chem_||_Year: 2023_||_Volu...,43,ChEMBL,CHEMBL5328243,20250102,20250102,...,NaN,485.0,40,NaN,NaN,NaN,10.1016/j.bmc.2023.117381,NaN,36018473.0,"Sasaki K, Takada H, Hayashi C, Ohya K, Yamaguc..."
229,1975742,Literature-derived,Antibacterial activity against azithromycin re...,Title: Synthesis of novobiocin derivatives and...,Journal: Bioorg Med Chem_||_Year: 2023_||_Volu...,43,ChEMBL,CHEMBL5328244,20250102,20250102,...,NaN,485.0,1,NaN,NaN,NaN,10.1016/j.bmc.2023.117381,NaN,36018473.0,"Sasaki K, Takada H, Hayashi C, Ohya K, Yamaguc..."
263,1975743,Literature-derived,Antibacterial activity against ceftriaxone int...,Title: Synthesis of novobiocin derivatives and...,Journal: Bioorg Med Chem_||_Year: 2023_||_Volu...,43,ChEMBL,CHEMBL5328245,20250102,20250102,...,NaN,485.0,1,NaN,NaN,NaN,10.1016/j.bmc.2023.117381,NaN,36018473.0,"Sasaki K, Takada H, Hayashi C, Ohya K, Yamaguc..."
296,1975744,Literature-derived,Antibacterial activity against Neisseria gonor...,Title: Synthesis of novobiocin derivatives and...,Journal: Bioorg Med Chem_||_Year: 2023_||_Volu...,43,ChEMBL,CHEMBL5328246,20250102,20250102,...,NaN,485.0,8,NaN,NaN,NaN,10.1016/j.bmc.2023.117381,NaN,36018473.0,"Sasaki K, Takada H, Hayashi C, Ohya K, Yamaguc..."


### 7.1 Where do they come from?

In [12]:
df_missing["aidsrcname"].value_counts()

aidsrcname
ChEMBL    574
Name: count, dtype: int64

In [17]:
df_missing["aidtype"].value_counts()

aidtype
Literature-derived    504
Confirmatory           70
Name: count, dtype: int64

### 7.2 Do they show Neisseria TaxIDs inside the `taxids` column?

In [ ]:
def extract_taxids(taxid_field):
    """Return a list of TaxIDs from the pipe-separated taxid field."""
    if pd.isna(taxid_field):
        return []
    return str(taxid_field).split("|")

def contains_neisseria(taxid_field):
    """Return True if any Neisseria taxid is in the field."""
    tid_list = extract_taxids(taxid_field)
    return any(t in tid_list for t in map(str, neisseria_taxids))

df_missing = df_missing.copy()
df_missing["parsed_taxids"] = df_missing["taxids"].apply(extract_taxids)
df_missing["contains_neisseria"] = df_missing["parsed_taxids"].apply(
    lambda lst: any(t in lst for t in map(str, neisseria_taxids))
)

df_missing["contains_neisseria"].value_counts()

contains_neisseria
False    520
True      54
Name: count, dtype: int64

### 7.3 Inspect the Neisseria-containing AIDs

In [27]:
df_missing_neis = df_missing[df_missing["contains_neisseria"]]

df_missing_neis[["aid", "aidsrcname", "aidname", "taxids","parsed_taxids"]].head(20)

,aid,aidsrcname,aidname,taxids,parsed_taxids
214,1982347,ChEMBL,Antimicrobial activity against Neisseria gonor...,485,[485]
218,1975741,ChEMBL,Antibacterial activity against resistant Neiss...,485,[485]
229,1975742,ChEMBL,Antibacterial activity against azithromycin re...,485,[485]
263,1975743,ChEMBL,Antibacterial activity against ceftriaxone int...,485,[485]
296,1975744,ChEMBL,Antibacterial activity against Neisseria gonor...,485,[485]
318,1880854,ChEMBL,Antimicrobial activity against wild type Neiss...,485,[485]
319,1880857,ChEMBL,Antimicrobial activity against wild type Neiss...,485,[485]
320,1880858,ChEMBL,Antimicrobial activity against Neisseria gonor...,485,[485]
332,1743155,ChEMBL,Antibacterial activity against Neisseria gonor...,485,[485]
333,1880849,ChEMBL,Antimicrobial activity against Neisseria gonor...,485,[485]


### 7.4 Inspect Non-Neisseria AIDs from the UI

In [28]:
df_missing_non = df_missing[~df_missing["contains_neisseria"]]

df_missing_non[["aid", "aidsrcname", "aidname", "parsed_taxids"]].head(20)

,aid,aidsrcname,aidname,parsed_taxids
365,530833,ChEMBL,Cmax in Neisseria gonorrhoeae infected patient...,[9606]
366,530835,ChEMBL,Protein binding in Neisseria gonorrhoeae infec...,[9606]
367,530834,ChEMBL,Half life in Neisseria gonorrhoeae infected pa...,[9606]
368,530832,ChEMBL,Oral bioavailability in Neisseria gonorrhoeae ...,[9606]
373,572147,ChEMBL,Antimicrobial activity against AcrAB-deficient...,[562]
383,58102,ChEMBL,In vitro inhibition of dihydrofolate reductase...,"[1582, 272623]"
420,1653739,ChEMBL,Ratio of IC50 for full-length human GCYH1A exp...,[]
496,231609,ChEMBL,Relative antimicrobial activity to that of tri...,[]
497,231610,ChEMBL,Relative antimicrobial activity to that of tri...,[]
498,231612,ChEMBL,Relative antimicrobial activity to that of tri...,[]


### 7.5 Check Whether These TaxIDs Belong to ANY of Your 15 Pathogens

In [31]:
# Load your pathogen taxonomy table
pathogen_tax_map = (
    tax_table.groupby("Pathogen")["Taxonomy_ID"]
    .apply(lambda x: set(map(str, x)))
    .to_dict()
)

def match_other_pathogens(tid_list):
    """Return pathogen names whose taxids appear in the list."""
    matches = []
    for pathogen, tids in pathogen_tax_map.items():
        if any(t in tid_list for t in tids):
            matches.append(pathogen)
    return matches

df_missing_non["other_pathogen_hits"] = df_missing_non["parsed_taxids"].apply(match_other_pathogens)

df_missing_non[["aid", "taxids", "parsed_taxids", "other_pathogen_hits"]].head(20)

/var/folders/xk/plgt8lts2_72l0rtt_cz7k6h0000gn/T/ipykernel_55382/597613131.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_missing_non["other_pathogen_hits"] = df_missing_non["parsed_taxids"].apply(match_other_pathogens)


,aid,taxids,parsed_taxids,other_pathogen_hits
365,530833,9606,[9606],[]
366,530835,9606,[9606],[]
367,530834,9606,[9606],[]
368,530832,9606,[9606],[]
373,572147,562,[562],[Escherichia coli]
383,58102,1582|272623,"[1582, 272623]",[]
420,1653739,NaN,[],[]
496,231609,NaN,[],[]
497,231610,NaN,[],[]
498,231612,NaN,[],[]


### 7.6. Check missing values in taxids

In [33]:
df_missing["taxids"].isna().sum()

28

### 7.6 Summary Statistics

In [35]:
summary_missing = pd.DataFrame({
    "Category": [
        "Missing AIDs (UI→Aid2Taxid)",
        "Contain Neisseria TaxID",
        "Match Other Pathogens",
        "Match No Known Pathogens",
        "Do Not Contain Any Taxid"
    ],
    "Count": [
        len(df_missing),
        df_missing["contains_neisseria"].sum(),
        sum(df_missing_non["other_pathogen_hits"].apply(len) > 0),
        sum(df_missing_non["other_pathogen_hits"].apply(len) == 0),
        sum(df_missing["taxids"].isna())
    ]
})

summary_missing

,Category,Count
0,Missing AIDs (UI→Aid2Taxid),574
1,Contain Neisseria TaxID,54
2,Match Other Pathogens,147
3,Match No Known Pathogens,373
4,Do Not Contain Any Taxid,28
